In [11]:
import numpy as np

In [12]:
def mlp_init(d0, d1, d2):
    """
    Initialize W1, b1, W2, b2
    d0: dimension of input data
    d1: number of hidden unit
    d2: number of output unit = number of classes
    """
    W1 = 0.01*np.random.randn(d0, d1)
    b1 = np.zeros(d1)
    W2 = 0.01*np.random.randn(d1, d2)
    b2 = np.zeros(d2)
    return (W1, b1, W2, b2)
def mlp_predict(X, W1, b1, W2, b2):
    """
    Suppose that the network has been trained, predict class of new points.
    X: data matrix, each ROW is one data point.
    W1, b1, W2, b2: learned weight matrices and biases
    """
    Z1 = X.dot(W1) + b1 # shape (N, d1) --> LR
    A1 = np.maximum(Z1, 0) # shape (N, d1) --> Use ReLU as activation fn
    Z2 = A1.dot(W2) + b2 # shape (N, d2) 
    return np.argmax(Z2, axis=1)

In [28]:
A = mlp_init(150,4,1)
A[3].shape

(1,)

In [13]:
def softmax_stable(Z):
    """
    Compute softmax values for each sets of scores in Z.
    each ROW of Z is a set of scores.
    """
    e_Z = np.exp(Z - np.max(Z, axis = 1, keepdims = True))
    A = e_Z / e_Z.sum(axis = 1, keepdims = True)
    return A
def crossentropy_loss(Yhat, y):
    """
    Yhat: a numpy array of shape (Npoints, nClasses) -- predicted output
    y: a numpy array of shape (Npoints) -- ground truth.
    NOTE: We don’t need to use the one-hot vector here since most of elements
    are zeros. When programming in numpy, in each row of Yhat, we need to access
    to the corresponding index only.
    """
    id0 = range(Yhat.shape[0])
    return -np.mean(np.log(Yhat[id0, y]))

In [79]:
def mlp_fit(X, y, W1, b1, W2, b2, eta):
    loss_hist = []
    for i in range(20000): # number of epoches
    # feedforward
        Z1 = X.dot(W1) + b1 # shape (N, d1)
        A1 = np.maximum(Z1, 0) # shape (N, d1)
        Z2 = A1.dot(W2) + b2 # shape (N, d2)
        Yhat = softmax_stable(Z2) # shape (N, d2)
        if i %5000 == 0: # print loss after each 1000 iterations
            loss = crossentropy_loss(Yhat, y)
            print("iter %d, loss: %f" %(i, loss))
            loss_hist.append(loss)
        # back propagation
        id0 = range(Yhat.shape[0])
        Yhat[id0, y] -=1
        E2 = Yhat/N # shape (N, d2)
        dW2 = np.dot(A1.T, E2) # shape (d1, d2)
        db2 = np.sum(E2, axis = 0) # shape (d2,)
        E1 = np.dot(E2, W2.T) # shape (N, d1)
        E1[Z1 <= 0] = 0 # gradient of ReLU, shape (N, d1)
        dW1 = np.dot(X.T, E1) # shape (d0, d1)
        db1 = np.sum(E1, axis = 0) # shape (d1,)
        # Gradient Descent update
        W1 += -eta*dW1
        b1 += -eta*db1
        W2 += -eta*dW2
        b2 += -eta*db2
    return (W1, b1, W2, b2, loss_hist)

In [57]:
from sklearn.datasets import load_iris
from sklearn import neighbors, datasets
np.random.seed(7)
iris = datasets.load_iris()
X = iris.data
y = iris.target
X.shape, y.shape
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [75]:

d0 = 4 # data dimension
d1 = h = 150 # number of hidden units
d2 = C = 3 # number of classes
eta = 1 # learning rate
(W1, b1, W2, b2) = mlp_init(d0, d1, d2)

In [80]:
W1
Z1 = X.dot(W1) + b1
A1 = np.maximum(Z1, 0)
Z2 = A1.dot(W2) + b2
Yhat = softmax_stable(Z2)
# Z1.shape, X.shape, W1.shape, b1.shape, A1.shape, W2.shape, b2.shape, 
# A1.shape, W2.shape, b2.shape, Yhat.shape, y.shape
N = 150
(W1_hat, b1_hat, W2_hat, b2_hat, loss_hist) = mlp_fit(X, y, W1, b1, W2, b2, eta)


iter 0, loss: 1.098612
iter 5000, loss: 1.098612
iter 10000, loss: 1.098612
iter 15000, loss: 1.098612


In [81]:
y_pred = mlp_predict(X, W1_hat, b1_hat, W2_hat, b2_hat)
np.mean(y_pred == y)

np.float64(0.3333333333333333)